<div style="border-left:6px solid #14b8a6;padding:18px 22px;background:#f7f7f7">
<p style="margin:0;color:#14b8a6;letter-spacing:2px;font-size:12px">DATA MINING E INTELIGENCIA DE NEGOCIOS · SESIÓN 5 · GUÍA 3</p>
<h1 style="margin:6px 0 4px 0;font-weight:300">¿Qué tipos de clientes tiene Mercado Central?</h1>
<p style="margin:0;font-size:16px;color:#444">Segmentación de clientes con <b>K-Means</b>, paso a paso, de los datos a la decisión.</p>
<p style="margin:10px 0 0 0;color:#777;font-size:13px">Profe Alejandra Otero · Universidad EAN · 2026</p>
</div>

<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/data-mining-36993/notebooks/Sesion5_KMeans_Mercado_Central.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

### Cómo usar este cuaderno

1. Si lo abriste desde el botón, ya estás en Colab. Ve a **Archivo → Guardar una copia en Drive** para tener tu propia versión.
2. Ejecuta cada celda de código con **Shift + Enter**, en orden, de arriba hacia abajo.
3. **No tienes que subir ningún archivo**: los datos se descargan solos desde GitHub.
4. Lee las celdas de texto: ahí está la explicación. Las celdas marcadas con **Tu turno** son para que pienses y respondas con tu grupo.

### Ruta de la sesión

| Paso | Qué hacemos | Para qué te sirve en la Guía 3 |
|---|---|---|
| 0 | Entender qué es agrupar | Base conceptual (Error frecuente 1) |
| 1 | Cargar y mirar los datos | Conocer la base depurada |
| 2 | Elegir **variables activas y pasivas** | Punto 5 de la guía |
| 3 | Estandarizar | Que ninguna variable "grite" más que las otras |
| 4 | Elegir el número de clústeres (k) | Punto 4: número de clústeres y **bajo qué criterio** |
| 5 | Entrenar K-Means y ver la **proporción** de cada grupo | Punto 4 |
| 6 | Perfilar los clústeres | Punto 6: perfiles |
| 7 | **Tablas de contingencia** con variables pasivas | Puntos 5 y 8: el Excel |
| 8 | Del clúster a la decisión | Insumo para la infografía (punto 7) |
| 9 | Exportar todo a Excel | Archivo que anexas |

## 0. Antes del código: ¿qué significa agrupar?

Hasta la Guía 2 hicimos **análisis descriptivo**: mirar variables una por una (univariado) y de a dos (bivariado). Ahora damos un salto: le pedimos a un algoritmo que **encuentre grupos por sí solo**.

**Aprendizaje supervisado vs. no supervisado**

| | Supervisado (Guía 4: árboles) | No supervisado (Guía 3: K-Means) |
|---|---|---|
| ¿Hay una respuesta correcta conocida? | Sí, una variable objetivo (por ejemplo, *respondió a la campaña*) | No. No hay etiqueta |
| ¿Qué hace el modelo? | Aprende a **predecir** esa respuesta | **Descubre** estructura: grupos de casos parecidos |
| Pregunta típica | ¿Este cliente va a comprar? | ¿Qué tipos de clientes tengo? |

**La idea de K-Means en una imagen mental.** Imagina que pones a los 2.030 clientes de Mercado Central en un salón, y cada cliente se para en un lugar según cuánto gana, cuánto gasta y cuántas veces compra. K-Means:

1. Pone **k banderas** (centroides) al azar en el salón.
2. Cada cliente camina hacia la bandera **más cercana**.
3. Cada bandera se mueve al **centro** de las personas que llegaron a ella.
4. Se repiten los pasos 2 y 3 hasta que las banderas ya no se mueven.

Al final, cada bandera es un **segmento** y su posición describe al "cliente típico" de ese segmento.

> **Lo que K-Means NO hace:** no te dice cuántos grupos hay (eso lo decides tú con criterios), ni le pone nombre a los grupos (eso es el trabajo del analista, y es lo que más vale en la Guía 3).

## 1. Preparar el entorno y cargar los datos

Primero importamos las **librerías**: cajas de herramientas que otras personas ya programaron.

| Librería | Alias | Para qué la usamos |
|---|---|---|
| `pandas` | `pd` | Tablas de datos (como Excel, pero en código) |
| `numpy` | `np` | Cálculos numéricos |
| `matplotlib` | `plt` | Gráficas |
| `scikit-learn` | `sklearn` | Los modelos de machine learning (K-Means, escaladores, métricas) |
| `scipy` | | Prueba estadística chi-cuadrado para las tablas de contingencia |

In [ ]:
# Importamos las librerías (Colab ya las tiene instaladas)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2_contingency

# Estilo de gráficas: escala de grises con un solo color de acento (turquesa)
ACENTO = "#14b8a6"
GRIS = "#9a9a9a"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#bbbbbb", "axes.titleweight": "bold", "axes.titlesize": 13,
    "axes.titlelocation": "left", "font.size": 11,
})
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
FUENTE = "Fuente: elaboración propia con la base Mercado Central (caso ficticio basado en Kaggle, Customer Personality Analysis)."
print("Listo: librerías cargadas")

Ahora cargamos la **base depurada** del curso (la que resultó del taller de limpieza de la Guía 1). Está guardada en GitHub, así que `pd.read_csv()` la descarga directamente desde internet. Nadie tiene que subir archivos.

In [ ]:
URL_DATOS = "https://raw.githubusercontent.com/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/main/data-mining-36993/data/mercado_central_depurada.csv"

df = pd.read_csv(URL_DATOS)        # lee el archivo y lo guarda en una tabla llamada df

print(f"La base tiene {df.shape[0]:,} clientes (filas) y {df.shape[1]} variables (columnas)")
df.head()                          # muestra las 5 primeras filas

**¿Qué estamos viendo?** Cada fila es un cliente. Hay variables de perfil (edad, ingreso, nivel educativo, hijos), de comportamiento de compra (gasto por categoría, número de compras por canal, días desde la última compra) y de respuesta a campañas.

La base ya viene depurada: sin faltantes, con edad, gasto total, compras totales e hijos totales ya calculados. Verifiquémoslo:

In [ ]:
print("Valores faltantes en toda la base:", df.isna().sum().sum())
df[["ingreso_anual_hogar", "gasto_total", "compras_total", "dias_desde_ultima_compra", "edad", "hijos_total"]].describe().T

## 2. Variables activas y variables pasivas

Este es **el concepto más importante de la Guía 3** (punto 5), y el que más se confunde.

| | Variables **activas** | Variables **pasivas** (o ilustrativas) |
|---|---|---|
| ¿Qué son? | Las que **entran al modelo** y definen los grupos | Las que **no entran** al modelo |
| ¿Para qué? | K-Means calcula distancias con ellas | Sirven **después** para describir y validar los grupos |
| Tipo | Numéricas (K-Means trabaja con distancias) | Pueden ser categóricas: nivel educativo, pareja, respuesta a campaña |
| ¿Cómo se analizan? | Con promedios por clúster | Con **tablas de contingencia** (clúster × categoría) |

**Criterio de negocio para elegir las activas.** Nuestra pregunta es *¿qué tipos de clientes tenemos según su valor y su forma de comprar?* Entonces elegimos variables de **capacidad** (ingreso), **valor** (gasto), **frecuencia** (compras), **recencia** (días desde la última compra), **contexto del hogar** (hijos), **sensibilidad al precio** (compras con descuento) y **relación digital** (visitas web). Es una idea muy cercana al modelo RFM (recencia, frecuencia, valor monetario) que se usa en mercadeo.

> **Tu turno 1.** Con tu grupo: si su pregunta de negocio fuera *¿qué clientes responden mejor a las campañas?*, ¿la variable `respondio_ultima_campana` debería ser activa o pasiva? ¿Por qué?

In [ ]:
# Variables ACTIVAS: entran al modelo
activas = [
    "ingreso_anual_hogar",       # capacidad de compra
    "gasto_total",               # valor del cliente
    "compras_total",             # frecuencia
    "dias_desde_ultima_compra",  # recencia (menos días = más reciente)
    "hijos_total",               # contexto del hogar
    "compras_con_descuento",     # sensibilidad al precio
    "visitas_web_mes",           # relación digital
]

# Variables PASIVAS: no entran, sirven para describir los grupos después
pasivas = ["nivel_educativo", "situacion_pareja", "respondio_ultima_campana", "edad"]

X = df[activas].copy()
X.describe().T[["mean", "std", "min", "max"]]

## 3. Estandarizar: que todas las variables hablen en el mismo idioma

Mira la tabla de arriba: el ingreso está en **decenas de miles**, y los hijos van de **0 a 3**. K-Means mide **distancias**, así que si no hacemos nada, el ingreso dominaría todo y los hijos no importarían.

La **estandarización (puntaje z)** convierte cada variable para que tenga promedio 0 y desviación estándar 1:

$$z = \frac{x - \text{promedio}}{\text{desviación estándar}}$$

Un z de **+1** significa "una desviación estándar por encima del promedio"; un z de **-1**, "una por debajo". Ahora todas las variables pesan lo mismo.

In [ ]:
escalador = StandardScaler()          # creamos el escalador
X_z = escalador.fit_transform(X)      # aprende promedio y desviación, y transforma

# Comprobación: promedio aprox. 0 y desviación aprox. 1 en todas
pd.DataFrame(X_z, columns=activas).describe().T[["mean", "std"]].round(2)

## 4. ¿Cuántos clústeres? Criterios para decidir k

K-Means necesita que **tú** le digas cuántos grupos buscar. La Guía 3 pide explicar **bajo qué criterio** elegiste k. Usamos cuatro métricas:

| Métrica | Qué mide | Cómo leerla |
|---|---|---|
| **Inercia** (WCSS) | Suma de distancias al cuadrado de cada cliente a su centroide | Siempre baja al subir k. Buscamos el **codo**: el punto donde deja de bajar mucho |
| **Coeficiente de silueta** | Qué tan cerca está cada cliente de su grupo frente al grupo vecino. Va de -1 a 1 | **Más alto es mejor**. Sobre 0,25 ya hay estructura razonable en datos reales de clientes |
| **Calinski-Harabasz** | Separación entre grupos / dispersión dentro de los grupos | **Más alto es mejor** |
| **Davies-Bouldin** | Parecido promedio entre cada grupo y su grupo más similar | **Más bajo es mejor** |

> Ninguna métrica decide sola. El criterio final combina **estadística + utilidad para el negocio**: ¿puede el equipo de mercadeo diseñar una acción distinta para cada grupo?

In [ ]:
resultados = []
for k in range(2, 9):
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42)
    etiquetas = modelo.fit_predict(X_z)
    resultados.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silueta": silhouette_score(X_z, etiquetas),
        "calinski_harabasz": calinski_harabasz_score(X_z, etiquetas),
        "davies_bouldin": davies_bouldin_score(X_z, etiquetas),
    })

metricas = pd.DataFrame(resultados).set_index("k")
metricas.style.format("{:,.2f}").highlight_max(subset=["silueta", "calinski_harabasz"], color="#ccf2ec") \
       .highlight_min(subset=["davies_bouldin"], color="#ccf2ec")

In [ ]:
K_ELEGIDO = 3   # <- la decisión del analista (ver explicación abajo)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
colores = [ACENTO if k == K_ELEGIDO else GRIS for k in metricas.index]

ax[0].plot(metricas.index, metricas["inercia"], color=GRIS, lw=1.5, zorder=1)
ax[0].scatter(metricas.index, metricas["inercia"], c=colores, s=70, zorder=2)
ax[0].set_title("El codo aparece en k = 3: después, cada grupo extra aporta poco")
ax[0].set_xlabel("Número de clústeres (k)"); ax[0].set_ylabel("Inercia")

ax[1].bar(metricas.index, metricas["silueta"], color=colores)
ax[1].set_title("k = 3 mantiene buena silueta y da grupos accionables")
ax[1].set_xlabel("Número de clústeres (k)"); ax[1].set_ylabel("Coeficiente de silueta")

fig.text(0, -0.04, FUENTE, fontsize=9, color="#777")
plt.tight_layout(); plt.show()

**Lectura de la decisión.** k = 2 tiene la silueta más alta, pero solo separa "clientes que gastan mucho" de "clientes que gastan poco": eso ya lo sabíamos con la Guía 2 y no le da al negocio nada nuevo. Con **k = 3** la curva de inercia hace el codo, la silueta sigue siendo razonable y aparece un tercer grupo con un comportamiento distinto (lo verás en el perfil). Con k = 4 o más, la silueta cae y los grupos se vuelven difíciles de diferenciar en una acción comercial.

**Así se escribe en el informe:** *"Se eligieron 3 clústeres combinando el método del codo (inercia), el coeficiente de silueta (0,30) y el criterio de accionabilidad: cada grupo admite una estrategia comercial diferente."*

> **Tu turno 2.** Si en la base de su organización la silueta más alta fuera k = 5, pero dos de esos grupos se ven casi iguales, ¿qué k elegirían y cómo lo justificarían?

## 5. Entrenar el modelo final y ver la proporción de cada grupo

Parámetros que usamos:

- `n_clusters=3`: cuántos grupos.
- `n_init=10`: K-Means arranca con banderas al azar; lo corre 10 veces y se queda con la mejor solución.
- `random_state=42`: fija el azar para que todas y todos obtengamos el mismo resultado.

Después **reordenamos** los clústeres de menor a mayor gasto, para que el número tenga sentido (1 = menor gasto).

In [ ]:
kmeans = KMeans(n_clusters=K_ELEGIDO, n_init=10, random_state=42)
etiqueta_original = kmeans.fit_predict(X_z)

# Reordenar: clúster 1 = menor gasto promedio ... clúster k = mayor gasto promedio
orden = pd.Series(df["gasto_total"].values).groupby(etiqueta_original).mean().sort_values().index
mapa = {viejo: nuevo + 1 for nuevo, viejo in enumerate(orden)}
df["cluster"] = pd.Series(etiqueta_original).map(mapa).values

tamanos = df["cluster"].value_counts().sort_index()
proporcion = pd.DataFrame({"clientes": tamanos, "proporcion_%": (tamanos / len(df) * 100).round(1)})
proporcion

## 6. Perfilar: ¿quién es cada clúster?

Calculamos el **promedio de cada variable activa por clúster**, en sus unidades originales (para que se entiendan) y en puntajes z (para comparar). Un mapa de calor de los z nos dice rápido qué hace distinto a cada grupo: **turquesa intenso = muy por encima del promedio**, gris = por debajo.

In [ ]:
perfil = df.groupby("cluster")[activas].mean()
perfil.insert(0, "clientes", tamanos)
perfil

In [ ]:
perfil_z = pd.DataFrame(X_z, columns=activas).groupby(df["cluster"].values).mean()

from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("gris_turquesa", ["#5f5f5f", "#f2f2f2", ACENTO])

fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.imshow(perfil_z.values, cmap=cmap, vmin=-1.5, vmax=1.5, aspect="auto")
ax.set_xticks(range(len(activas))); ax.set_xticklabels([a.replace("_", "\n") for a in activas], fontsize=9)
ax.set_yticks(range(len(perfil_z))); ax.set_yticklabels([f"Clúster {c}" for c in perfil_z.index])
for i in range(perfil_z.shape[0]):
    for j in range(perfil_z.shape[1]):
        ax.text(j, i, f"{perfil_z.values[i, j]:+.1f}", ha="center", va="center", fontsize=9)
ax.set_title("Valor, hijos y descuento separan a los clientes; la recencia no los diferencia")
for s in ax.spines.values(): s.set_visible(False)
fig.text(0, -0.22, FUENTE + " Valores en puntaje z (0 = promedio de la base).", fontsize=9, color="#777")
plt.show()

### Ponerle nombre a cada grupo (esto es lo que más vale)

El nombre de un segmento debe resumir su comportamiento y sugerir una acción. Lee el mapa de calor y completa con tu grupo. Abajo te dejo la propuesta para la base del curso; **compárala con lo que tú ves**.

Un truco: describe cada clúster con la fórmula **"Son clientes que ___, ___ y ___; por eso necesitan ___"**.

In [ ]:
# Propuesta de nombres para la base del curso (clúster 1 = menor gasto)
nombres = {
    1: "Familias cuidando el bolsillo",
    2: "Cazadores de ofertas en crecimiento",
    3: "Clientes premium de alto valor",
}
df["segmento"] = df["cluster"].map(nombres)

resumen = df.groupby("segmento").agg(
    clientes=("id_cliente", "count"),
    ingreso_prom=("ingreso_anual_hogar", "mean"),
    gasto_prom=("gasto_total", "mean"),
    compras_prom=("compras_total", "mean"),
    hijos_prom=("hijos_total", "mean"),
    descuento_prom=("compras_con_descuento", "mean"),
    visitas_web=("visitas_web_mes", "mean"),
).sort_values("gasto_prom")
resumen["proporcion_%"] = (resumen["clientes"] / len(df) * 100).round(1)
resumen

## 7. Tablas de contingencia con las variables pasivas

Una **tabla de contingencia** cruza dos variables categóricas y cuenta cuántos casos caen en cada combinación. Aquí cruzamos **clúster × variable pasiva**. Es exactamente lo que pide el punto 5 de la guía, y lo que va en el Excel.

Dos formas de leerla:

- **Conteos**: cuántos clientes de cada clúster están en cada categoría.
- **Porcentaje por fila**: de cada clúster, qué % está en cada categoría. **Esta es la que se interpreta**, porque los clústeres tienen tamaños distintos.

**Prueba chi-cuadrado (χ²).** Responde: *¿la distribución de la variable pasiva cambia según el clúster, o es igual en todos?* Si el **p-valor < 0,05**, decimos que hay una **asociación estadísticamente significativa**: la variable pasiva ayuda a describir los segmentos. Esto le da el "fundamento estadístico" que pide el Error frecuente 4.

In [ ]:
def tabla_contingencia(variable_pasiva, etiqueta=None):
    conteo = pd.crosstab(df["segmento"], df[variable_pasiva], margins=True, margins_name="Total")
    pct_fila = pd.crosstab(df["segmento"], df[variable_pasiva], normalize="index").mul(100).round(1)
    chi2, p, gl, _ = chi2_contingency(pd.crosstab(df["segmento"], df[variable_pasiva]))
    print(f"\n=== Clúster × {etiqueta or variable_pasiva} ===")
    print(f"Chi-cuadrado = {chi2:,.1f} | grados de libertad = {gl} | p-valor = {p:.4f}",
          "-> asociación significativa" if p < 0.05 else "-> sin asociación significativa")
    display(pct_fila.style.format("{:.1f} %").background_gradient(cmap="Greys", axis=1))
    return conteo, pct_fila, p

# Creamos rangos de edad y una variable de respuesta legible
df["rango_edad"] = pd.cut(df["edad"], bins=[17, 30, 45, 60, 80], labels=["18-30", "31-45", "46-60", "61+"])
df["respuesta_campana"] = df["respondio_ultima_campana"].map({1: "Respondió", 0: "No respondió"})

tablas = {}
for var, nombre in [("nivel_educativo", "nivel educativo"), ("situacion_pareja", "situación de pareja"),
                    ("rango_edad", "rango de edad"), ("respuesta_campana", "respuesta a la última campaña")]:
    tablas[nombre] = tabla_contingencia(var, nombre)

In [ ]:
# La pasiva más interesante para el negocio: ¿quién responde a las campañas?
tasa = df.groupby("segmento")["respondio_ultima_campana"].mean().mul(100).sort_values()
colores = [ACENTO if v == tasa.max() else GRIS for v in tasa.values]

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.barh(tasa.index, tasa.values, color=colores)
for i, v in enumerate(tasa.values):
    ax.text(v + 0.5, i, f"{v:.1f} %", va="center")
ax.set_title(f"Los {tasa.idxmax().lower()} responden a las campañas mucho más que el resto")
ax.set_xlabel("% de clientes que respondió a la última campaña")
ax.spines["left"].set_visible(False)
fig.text(0, -0.06, FUENTE, fontsize=9, color="#777")
plt.show()

> **Tu turno 3.** Mira la tabla de **rango de edad**. Recuerda que la edad no entró al modelo. ¿Aun así quedó asociada con los clústeres? ¿Qué significa eso para el negocio?

## 8. Ver los grupos en un plano (opcional, pero muy útil para la infografía)

Tenemos 7 variables activas, así que no podemos dibujar 7 dimensiones. El **Análisis de Componentes Principales (PCA)** resume las 7 en 2 ejes que conservan la mayor parte de la información. No es parte de la guía; es una ayuda visual.

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_z)
var_exp = pca.explained_variance_ratio_.sum() * 100

fig, ax = plt.subplots(figsize=(9, 5.5))
tonos = {1: "#cfcfcf", 2: "#7d7d7d", 3: ACENTO}
for c in sorted(df["cluster"].unique()):
    m = df["cluster"].values == c
    ax.scatter(coords[m, 0], coords[m, 1], s=10, alpha=0.6, color=tonos.get(c, GRIS), label=nombres[c])
ax.set_title("Los tres segmentos ocupan zonas distintas del mapa de clientes")
ax.set_xlabel("Componente 1 (sobre todo valor: ingreso, gasto y compras)"); ax.set_ylabel("Componente 2")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
fig.text(0, -0.03, FUENTE + f" Los dos ejes conservan el {var_exp:.0f} % de la información.", fontsize=9, color="#777")
plt.show()

## 9. Del clúster a la decisión

Un segmento sin acción es solo una tabla. Esta es la parte que convierte el modelo en **inteligencia de negocios**, y es la que debe brillar en la infografía.

| Segmento | Quiénes son (según el perfil de la base del curso) | Acción sugerida | Métrica para medir |
|---|---|---|---|
| **Familias cuidando el bolsillo** (46 %) | Menor ingreso (≈ 35 mil) y gasto muy bajo (≈ 105), pocas compras, 1,2 hijos en promedio; visitan mucho la web pero compran poco; los más jóvenes | Convertir la visita web en compra: canasta familiar con precio fijo y envío gratis desde un monto bajo | Tasa de conversión web y frecuencia de compra |
| **Cazadores de ofertas en crecimiento** (23 %) | Ingreso medio (≈ 57 mil), gasto medio (≈ 737), compran seguido y son los que **más usan descuentos** (4,6 compras); hogares con hijos; 31 a 60 años | Programa de puntos y cupones personalizados para subirlos a premium sin regalar margen | Ticket promedio y % de compras con descuento |
| **Clientes premium de alto valor** (30 %) | Mayor ingreso (≈ 74 mil) y gasto (≈ 1.275), casi sin hijos, poco descuento y poca visita web; **responden más a campañas (25 %)** | Lanzamientos premium, experiencias y atención preferente; son el público de las campañas | Retención y tasa de respuesta a campañas |

**Un hallazgo que vale oro:** los *días desde la última compra* salen casi en 0 (el promedio) en los tres clústeres. Es decir, la **recencia no diferencia** a los grupos en esta base. Y en las tablas de contingencia, la **situación de pareja tampoco** (p-valor > 0,05). Reportar lo que **no** separa a los grupos también es análisis, y es una buena razón para revisar las variables activas.

> **Tu turno 4.** Esta tabla es una propuesta. Verifíquenla con los números de las secciones 6 y 7: ¿cambiarían algún nombre o alguna acción?

**Checklist de la Guía 3** (marca lo que ya tienes):

- [ ] Objetivos ajustados con la retroalimentación de la Guía 2 (punto 1)
- [ ] Número de clústeres, **criterio** y **proporción** de cada grupo (punto 4)
- [ ] Variables activas y pasivas, con **tablas de contingencia** (punto 5)
- [ ] Perfil y nombre de cada clúster (punto 6)
- [ ] Infografía con los perfiles (punto 7)
- [ ] Excel con las tablas de contingencia (punto 8)

## 10. Exportar todo a Excel para tu entrega

Esta celda crea un archivo Excel con una hoja por resultado: la base con el clúster de cada cliente, las métricas para elegir k, la proporción, el perfil y cada tabla de contingencia. En Colab se descarga solo; si no, búscalo en el panel de archivos (icono de carpeta, a la izquierda).

In [ ]:
archivo = "Guia3_KMeans_Mercado_Central.xlsx"
with pd.ExcelWriter(archivo, engine="openpyxl") as w:
    df.to_excel(w, sheet_name="base_con_cluster", index=False)
    metricas.to_excel(w, sheet_name="criterio_k")
    proporcion.to_excel(w, sheet_name="proporcion")
    resumen.to_excel(w, sheet_name="perfil_segmentos")
    for nombre, (conteo, pct, p) in tablas.items():
        hoja = ("ct_" + nombre.replace(" ", "_"))[:31]
        conteo.to_excel(w, sheet_name=hoja, startrow=1)
        pct.to_excel(w, sheet_name=hoja, startrow=len(conteo) + 5)
        ws = w.sheets[hoja]
        ws.cell(row=1, column=1, value=f"Conteos  (chi-cuadrado p-valor = {p:.4f})")
        ws.cell(row=len(conteo) + 5, column=1, value="Porcentaje por fila (%)")
print("Archivo creado:", archivo)

try:
    from google.colab import files
    files.download(archivo)
except ImportError:
    pass

## Para llevar

- **K-Means descubre grupos**; el analista decide **cuántos** y les da **sentido**.
- Las **variables activas** construyen los grupos; las **pasivas** los describen y validan con **tablas de contingencia** y chi-cuadrado.
- Siempre **estandariza** antes de usar distancias.
- El valor no está en el código: está en el **nombre, el perfil y la acción** de cada segmento.

**Si trabajas con los datos de tu propia organización:** cambia `URL_DATOS` por tu archivo (súbelo al panel de archivos de Colab y usa `pd.read_csv("tu_archivo.csv")`) y ajusta las listas `activas` y `pasivas` en la sección 2. El resto del cuaderno funciona igual.

**Lectura de la guía:** Gironés, J., Casas, J., Minguillón, J. y Caihuelas, R. (2017). *Minería de datos: modelos y algoritmos*. Editorial UOC. Capítulo 8.

<p style="color:#777;font-size:13px">¿Dudas? Escríbeme por el Inbox de Canvas o tráelas a la tutoría del jueves. Profe Alejandra Otero.</p>